Tools
Models can request to call tools that performs tasks such as fetching a data from database , searching the web or running a code. tools are pairings of:
  1.A schema,including name of a tool , a description, and definitions(often a JSON schema)
  2.A function or coroutine to execute
  

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("write something about langchain")
response.text


'<think>\nOkay, the user wants me to write about LangChain. Let me start by recalling what LangChain is. It\'s a framework for developing applications powered by large language models (LLMs). I should explain its main purpose and key components.\n\nFirst, I need to mention the core features. LangChain has modules like prompts, models, memory, and agents. Each of these plays a role in building and managing LLM applications. The prompts module helps in creating and managing prompts, which are essential for interacting with LLMs effectively.\n\nThen, the models module allows integration with various LLMs like OpenAI, Hugging Face, etc. That\'s important because it\'s about flexibility in choosing the underlying model. The memory module is for state management, which is crucial for maintaining context across multiple interactions, especially in chatbots or conversational agents.\n\nAgents are another key part. They enable the LLM to decide which tools to use, making the application more dy

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get the current weather in a given location"""
    return "The current weather in {} is sunny with a high of 25 degrees".format(location)

model_with_tools = model.bind_tools([get_weather])


In [7]:
response=model_with_tools.invoke("what is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(tool_call['name'])
    print(tool_call['args'])

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. I need to use the get_weather function. Let me check the function parameters. The function requires a location, which is provided as New York. So I\'ll call get_weather with location set to "New York". Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'z0g43mvrg', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 156, 'total_tokens': 250, 'completion_time': 0.147205683, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007036543, 'prompt_tokens_details': None, 'queue_time': 0.046466316, 'total_time': 0.154242226}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='

Tool execution loops


In [9]:
messages=[{"role": "user", "content": "what is the weather in New York?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
    
final_response=model_with_tools.invoke(messages)
print(final_response.text)
   

The current weather in New York is sunny, with a high temperature of 25°C. Let me know if you need more details! ☀️


//this is the manual working of create_agent

In [10]:
messages

[{'role': 'user', 'content': 'what is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. The user provided "New York" as the location. So I should call get_weather with location set to "New York". I\'ll make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': '38mzwjc66', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 156, 'total_tokens': 262, 'completion_time': 0.165176969, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.007723933, 'prompt_tokens_details': None, 'queue_time': 0.054433313, 'total_time': 0.172900902}, 'model_name': 'qwen/qwen3-32b', 'system_fingerp